# SkinAI — DenseNet121 분류 모델 학습

**Google Colab / 로컬 환경 모두 지원**
- Colab: Drive 마운트 → 레포 클론 → 심링크
- 로컬: 현재 디렉토리 그대로 사용

In [1]:
# GPU model check
!nvidia-smi

Wed Apr 22 14:28:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# 시스템 RAM 확인
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"현재 할당된 총 시스템 RAM: {ram_gb:.2f} GB")

현재 할당된 총 시스템 RAM: 179.37 GB


In [3]:
# ── 셀 1: 환경 감지 ────────────────────────────────────────────
import os
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
    # Colab 런타임 임시 경로 (세션 종료 시 소실)
    COLAB_ROOT = "/content/colab_skin_ai"
    PROJECT_ROOT = COLAB_ROOT
except ImportError:
    IS_COLAB = False
    # 로컬: 노트북이 위치한 레포 루트 그대로 사용
    LOCAL_ROOT = str(Path.cwd())
    PROJECT_ROOT = LOCAL_ROOT

print(f"환경       : {'Google Colab' if IS_COLAB else '로컬'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

환경       : Google Colab
PROJECT_ROOT: /content/colab_skin_ai


In [4]:
# ── 셀 2: Drive 마운트 (Colab 전용) ────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Drive 마운트 경로 (Colab 런타임 전용)
    DRIVE_ROOT = "/content/drive/MyDrive/skin_ai"
else:
    print("로컬 환경 — Drive 마운트 건너뜀")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# ── 셀 3: 소스코드 clone / pull (Colab 전용) ────────────────────
if IS_COLAB:
    from dotenv import load_dotenv

    # Drive에 저장된 .env에서 GITHUB_TOKEN 로드
    _env_path = f"{DRIVE_ROOT}/.env"
    if Path(_env_path).exists():
        load_dotenv(_env_path)
    else:
        print(f"경고: {_env_path} 없음 — GITHUB_TOKEN 없이 시도합니다")

    _token = os.getenv("GITHUB_TOKEN", "")
    _repo_url = (
        f"https://{_token}@github.com/kyoe-23/skin_ai.git"
        if _token else
        "https://github.com/kyoe-23/skin_ai.git"
    )

    if not Path(COLAB_ROOT).exists():
        !git clone {_repo_url} {COLAB_ROOT}
    else:
        !git -C {COLAB_ROOT} pull
else:
    print("로컬 환경 — 클론 건너뜀")


remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 554 bytes | 554.00 KiB/s, done.
From https://github.com/kyoe-23/skin_ai
   0403f56..5370de6  main       -> origin/main
Updating 0403f56..5370de6
Fast-forward
 ai/testing/evaluate.py      | 1 +
 ai/testing/threshold_opt.py | 1 +
 2 files changed, 2 insertions(+)


In [6]:
# ── 셀 4: 프로젝트 루트 이동 + 데이터 경로 설정 ────────────────
os.chdir(PROJECT_ROOT)
print(f"현재 디렉토리: {os.getcwd()}")

if IS_COLAB:
    # Colab 런타임: Drive에 있는 dataset_14를 심링크로 연결
    os.makedirs("data", exist_ok=True)
    DRIVE_DATASET = f"{DRIVE_ROOT}/data/dataset_14"
    if Path(DRIVE_DATASET).exists():
        !ln -sfn {DRIVE_DATASET} data/dataset_14
        print("data/dataset_14 심링크 생성 완료")
    else:
        print(f"경고: Drive 데이터셋 없음 — {DRIVE_DATASET}")
else:
    # 로컬: data/dataset_14 존재 여부만 확인
    if not Path("data/dataset_14").exists():
        print("경고: data/dataset_14/ 없음 — AI Hub 데이터를 먼저 배치하세요")
    else:
        print("data/dataset_14/ 확인 완료")

!ls data/

현재 디렉토리: /content/colab_skin_ai
data/dataset_14 심링크 생성 완료
dataset_14  processed


In [7]:
# ── 셀 5: 패키지 설치 ───────────────────────────────────────────
!pip install -q \
    torch torchvision \
    pandas pillow tqdm \
    matplotlib python-dotenv scikit-learn

# Grad-CAM (추론 서버 사용 시)
# !pip install -q pytorch-grad-cam

In [ ]:
# ── 셀 6: 학습 실행 ─────────────────────────────────────────────
# CSV의 zip_path(로컬 절대경로)를 --root_dir로 실행 환경 경로에 맞게 재매핑

# DenseNet121 (기본)
!python -m ai.training.classifier.train \
    --backbone densenet121 \
    --num_epochs 100 \
    --batch_size 64 \
    --root_dir {PROJECT_ROOT}

# !CHECKPOINT_DIR=ai/results_efficientnet \
#    python -m ai.training.classifier.train \
#    --backbone efficientnet_b3 \
#    --num_epochs 100 \
#    --batch_size 64 \
#    --root_dir {PROJECT_ROOT}

# 세션 만료 후 재개
# !python -m ai.training.classifier.train \
#    --backbone densenet121 \
#    --num_epochs 100 \
#    --batch_size 64 \
#    --resume ai/results/epoch_N.pth \
#    --root_dir {PROJECT_ROOT}

INFO [INFO] CUDA 사용
AI Hub 08-14 분류 모델 학습
  backbone : densenet121
  device   : cuda
  epochs   : 100
  batch    : 64
  lr       : 0.0005
  warmup   : 3 epochs
  scheduler: CosineAnnealingLR (T_max=97)
INFO Dataset 로드: data/processed/train.csv (9600건, direction=None)
INFO Dataset 로드: data/processed/val.csv (1200건, direction=None)

  Train: 9600건
  Val  : 1200건
INFO   Model     : DenseNet
INFO   Total     : 6,960,006 params
INFO   Trainable : 6,960,006 params

학습 시작...

[Epoch 01/100] Train Loss: 0.994 | Train Top-1: 72.9%
                    Val   Loss: 0.741 | Val   Top-1: 85.2% | Val Top-3: 97.6%
                    시간: 140.5s | LR: 0.000500
INFO   체크포인트 저장: ai/results/best.pth
                    Best 모델 저장 (Top-1: 85.17%)

[Epoch 02/100] Train Loss: 0.771 | Train Top-1: 84.6%
                    Val   Loss: 0.788 | Val   Top-1: 83.3% | Val Top-3: 97.3%
                    시간: 103.0s | LR: 0.000500

[Epoch 03/100] Train Loss: 0.720 | Train Top-1: 87.2%
                    Val   Loss

In [13]:
# ── 셀 7: 평가 + Threshold 최적화 ──────────────────────────────
!python -m ai.testing.evaluate \
    --checkpoint ai/results/best.pth \
    --root_dir {PROJECT_ROOT}

!python -m ai.testing.threshold_opt \
    --checkpoint ai/results/best.pth \
    --root_dir {PROJECT_ROOT}

INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/val.csv (1200건, direction=None)
평가 데이터: val.csv (1200건)

 SkinAI 분류 모델 평가 결과 (AI Hub 08-14)
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 92.17%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 99.00%
 Macro F1-Score : 0.9222
 Macro AUC      : 0.9921
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 건선             0.9465   0.8850   0.9147   0.9893
 아토피피부염         0.9494   0.8450   0.8942   0.9887
 여드름            0.9366   0.9600   0.9481   0.9968
 주사             0.9289   0.9150   0.9219   0.9950
 지루피부염          0.7983   0.9300   0.8591   0.9826
 정상             0.9950   0.9950   0.9950   1.0000
/content/colab_skin_ai/ai/testing/evaluate.py:115: UserWarning: Glyph 44148 (\N{HANGUL SYLLABLE GEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/content/colab_skin_ai/ai/tes

In [15]:
# ── 셀 8: 체크포인트 Drive 저장 (Colab 전용) ────────────────────────
# 런타임 종료 전 반드시 실행 — 저장하지 않으면 학습 결과 소실
if IS_COLAB:
    import shutil
    from pathlib import Path

    # 실제 존재하는 체크포인트 경로 자동 감지
    CANDIDATE_DIRS = [
        f"{COLAB_ROOT}/ai/results",
        f"{COLAB_ROOT}/ai/checkpoints/aihub",
    ]
    CKPT_SRC = next((d for d in CANDIDATE_DIRS if Path(d).exists()), None)
    if CKPT_SRC is None:
        raise FileNotFoundError("체크포인트 디렉토리를 찾을 수 없습니다.")

    CKPT_DST = f"{DRIVE_ROOT}/ai/results"
    shutil.copytree(CKPT_SRC, CKPT_DST, dirs_exist_ok=True)
    print(f"Drive 저장 완료: {CKPT_DST}")
else:
    print("로컬 환경 — 체크포인트 이미 ai/results/ 에 저장됨")

Drive 저장 완료: /content/drive/MyDrive/skin_ai/ai/results
